In [1]:
import os, re, time, subprocess, pythoncom, psutil, tempfile
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from IPython.display import HTML, display

# ══════════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════════
first_glob   = os.path.expanduser("~").replace("\\", "/")
base_path    = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources"
ADH_PATH     = f"{base_path}/Schedule_Adherence_Gap.parquet"
REST_PATH    = f"{base_path}/Agent_Rest_Compliance_Log.parquet"

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — EMAIL
# ══════════════════════════════════════════════════════════════════════════════
DISPLAY_NOTEBOOK = True
SEND_EMAIL       = True
ATTACH_RAWDATA   = True
ATTACH_FORMAT    = "xlsx"

EMAIL_TO = (
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com"
)
 
EMAIL_CC = (
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "van.tran@concentrix.com;"
    "duonghoangvu.pham@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com;"
    "atul.pathak@concentrix.com"
)

OVERRIDE_DATE = None   # None = auto D-1 | "2026-05-29"

if OVERRIDE_DATE:
    report_date = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
else:
    report_date = datetime.now() - timedelta(days=1)

ACT_COLS          = ["Break","Lunch","Training | Coaching"]
REST_SESSION_COLS = ["Break_1","Break_2","Lunch"]
EXCEED_BUCKETS    = ["< 1 min","1-5 min","5-10 min","> 10 min"]

ADH_GREEN  = 85.0
ADH_YELLOW = 75.0

report_date_d = report_date.date()
report_now    = datetime.now()
current_month = report_date.strftime("%y_%m")
month_start   = report_date.replace(day=1).date()

def _ordinal(n):
    s = {1:"st",2:"nd",3:"rd"}.get(n%10 if n%100 not in (11,12,13) else 0,"th")
    return f"{n}{s}"

day_ord       = _ordinal(report_date.day)
month_yr      = report_date.strftime("%b'%y")
report_date_s = report_date.strftime("%d-%b-%Y")
EMAIL_SUBJECT = f"Expedia VN - Agent Compliance Report - as of the {day_ord} of {month_yr}"

print(f"✓ Report date : {report_date_d}")
print(f"✓ Month       : {current_month}")
print(f"✓ Subject     : {EMAIL_SUBJECT}")

# ══════════════════════════════════════════════════════════════════════════════
# LOB MAPPING
# ══════════════════════════════════════════════════════════════════════════════
def map_lob(lob):
    if pd.isna(lob): return None
    s = str(lob).strip()
    if s == "Support_LG_Nesting":  return "Lodging"
    if s == "Support_NL_Nesting":  return "Non_Lodging"
    if s == "Lodging":             return "Lodging"
    if s == "Lodging_Nesting":     return "Lodging"
    if s == "Non_Lodging":         return "Non_Lodging"
    if s == "Non_Lodging_Nesting": return "Non_Lodging"
    return None

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & PREPARE — Schedule Adherence Gap
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading Schedule_Adherence_Gap...")
adh_raw = (
    pl.read_parquet(ADH_PATH)
    .with_columns(pl.col("Date").cast(pl.Date, strict=False))
    .filter(
        (pl.col("Month") == pl.lit(current_month)) &
        (pl.col("Date") >= pl.lit(month_start)) &
        (pl.col("Date") <= pl.lit(report_date_d))
    )
    .to_pandas()
)
adh_raw["LOB"] = adh_raw["LOB"].apply(map_lob)
adh = adh_raw[adh_raw["Scheduled_Duration_Mins"] > 0].copy()
print(f"✓ Adherence rows MTD to {report_date_d}: {len(adh)}")

def calc_adh_rate(df, group_cols):
    if group_cols:
        g = df.groupby(group_cols).agg(
            actual   =("Actual_Duration_Mins",    "sum"),
            scheduled=("Scheduled_Duration_Mins", "sum")
        ).reset_index()
    else:
        g = pd.DataFrame([{
            "actual"   : df["Actual_Duration_Mins"].sum(),
            "scheduled": df["Scheduled_Duration_Mins"].sum(),
        }])
    g["Rate"] = np.where(
        g["scheduled"] > 0,
        (g["actual"] / g["scheduled"] * 100).round(1),
        np.nan
    )
    return g

def build_adh_pivot(df, index_col, sort_map=None):
    by_act = calc_adh_rate(df, [index_col, "Activity_Type"])
    pivot  = by_act.pivot_table(
        index=index_col, columns="Activity_Type", values="Rate"
    ).reset_index()
    for c in ACT_COLS:
        if c not in pivot.columns: pivot[c] = np.nan
    gt_idx = calc_adh_rate(df, [index_col])[[index_col,"Rate"]].rename(
        columns={"Rate":"Grand Total"})
    pivot  = pivot.merge(gt_idx, on=index_col, how="left")
    pivot  = pivot[[index_col] + ACT_COLS + ["Grand Total"]]
    gt_act = calc_adh_rate(df, ["Activity_Type"])
    gt_dict= dict(zip(gt_act["Activity_Type"], gt_act["Rate"]))
    gt_all = calc_adh_rate(df, []).iloc[0]["Rate"]
    gt_row = {index_col: "Grand Total"}
    for c in ACT_COLS: gt_row[c] = gt_dict.get(c, np.nan)
    gt_row["Grand Total"] = gt_all
    pivot  = pd.concat([pivot, pd.DataFrame([gt_row])], ignore_index=True)
    if sort_map:
        pivot["_ord"] = pivot[index_col].map(sort_map).fillna(99)
        pivot = pivot.sort_values("_ord").drop(columns=["_ord"])
    return pivot

adh_sup = build_adh_pivot(adh, "Supervisor Name")
adh_lob = build_adh_pivot(adh, "LOB", sort_map={"Lodging":0,"Non_Lodging":1})

adh_day_by_act = calc_adh_rate(adh, ["Date","Activity_Type"])
adh_day_pvt    = adh_day_by_act.pivot_table(
    index="Date", columns="Activity_Type", values="Rate"
).reset_index()
adh_day_pvt["Date"] = pd.to_datetime(adh_day_pvt["Date"]).dt.strftime("%Y-%m-%d")
for c in ACT_COLS:
    if c not in adh_day_pvt.columns: adh_day_pvt[c] = np.nan

adh_day_gt = calc_adh_rate(adh, ["Date"])
adh_day_gt["Date"] = adh_day_gt["Date"].dt.strftime("%Y-%m-%d")
adh_day_pvt = adh_day_pvt.merge(
    adh_day_gt[["Date","Rate"]].rename(columns={"Rate":"Grand Total"}),
    on="Date", how="left"
).sort_values("Date")

gt_act_all  = calc_adh_rate(adh, ["Activity_Type"])
gt_act_dict = dict(zip(gt_act_all["Activity_Type"], gt_act_all["Rate"]))
gt_all_day  = calc_adh_rate(adh, []).iloc[0]["Rate"]
gt_day_row  = {"Date":"Grand Total"}
for c in ACT_COLS: gt_day_row[c] = gt_act_dict.get(c, np.nan)
gt_day_row["Grand Total"] = gt_all_day
adh_day_pvt = pd.concat([adh_day_pvt, pd.DataFrame([gt_day_row])], ignore_index=True)

print(f"✓ Adherence pivots — Sup:{len(adh_sup)} LOB:{len(adh_lob)} Day:{len(adh_day_pvt)}")

# ══════════════════════════════════════════════════════════════════════════════
# PREPARE — Agent Rest Compliance
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading Agent_Rest_Compliance_Log...")
rest_raw = (
    pl.read_parquet(REST_PATH)
    .with_columns(pl.col("Date").cast(pl.Date, strict=False))
    .filter(
        (pl.col("Month") == pl.lit(current_month)) &
        (pl.col("Date") >= pl.lit(month_start)) &
        (pl.col("Date") <= pl.lit(report_date_d))
    )
    .to_pandas()
)
rest_raw["LOB"] = rest_raw["LOB"].apply(map_lob)
rest_exceed = rest_raw[rest_raw["Is_Exceeding_Limit"] != "No"].copy()
print(f"✓ Rest exceeding rows MTD to {report_date_d}: {len(rest_exceed)}")

def build_rest_pivot(df, index_col, sort_map=None):
    g = (df.groupby([index_col,"Rest_Session_Name"])
           .size().reset_index(name="Count"))
    pivot = g.pivot_table(
        index=index_col, columns="Rest_Session_Name",
        values="Count", fill_value=0
    ).reset_index()
    for c in REST_SESSION_COLS:
        if c not in pivot.columns: pivot[c] = 0
    pivot = pivot[[index_col] + REST_SESSION_COLS]
    pivot["Grand Total"] = pivot[REST_SESSION_COLS].sum(axis=1)
    gt_row = {index_col:"Grand Total"}
    for c in REST_SESSION_COLS: gt_row[c] = int(pivot[c].sum())
    gt_row["Grand Total"] = int(pivot["Grand Total"].sum())
    pivot = pd.concat([pivot, pd.DataFrame([gt_row])], ignore_index=True)
    if sort_map:
        pivot["_ord"] = pivot[index_col].map(sort_map).fillna(99)
        pivot = pivot.sort_values("_ord").drop(columns=["_ord"])
    return pivot

def build_bucket_pivot(df, index_col, sort_map=None):
    g = (df.groupby([index_col,"Exceed_Bucket"])
           .size().reset_index(name="Count"))
    pivot = g.pivot_table(
        index=index_col, columns="Exceed_Bucket",
        values="Count", fill_value=0
    ).reset_index()
    for c in EXCEED_BUCKETS:
        if c not in pivot.columns: pivot[c] = 0
    pivot = pivot[[index_col] + EXCEED_BUCKETS]
    pivot["Grand Total"] = pivot[EXCEED_BUCKETS].sum(axis=1)
    gt_row = {index_col:"Grand Total"}
    for c in EXCEED_BUCKETS: gt_row[c] = int(pivot[c].sum())
    gt_row["Grand Total"] = int(pivot["Grand Total"].sum())
    pivot = pd.concat([pivot, pd.DataFrame([gt_row])], ignore_index=True)
    if sort_map:
        pivot["_ord"] = pivot[index_col].map(sort_map).fillna(99)
        pivot = pivot.sort_values("_ord").drop(columns=["_ord"])
    return pivot

def build_bucket_session_pivot(df):
    g = (df.groupby(["Exceed_Bucket","Rest_Session_Name"])
           .size().reset_index(name="Count"))
    pivot = g.pivot_table(
        index="Exceed_Bucket", columns="Rest_Session_Name",
        values="Count", fill_value=0
    ).reset_index()
    for c in REST_SESSION_COLS:
        if c not in pivot.columns: pivot[c] = 0
    pivot = pivot[["Exceed_Bucket"] + REST_SESSION_COLS]
    pivot["Grand Total"] = pivot[REST_SESSION_COLS].sum(axis=1)
    bucket_order = {"< 1 min":0,"1-5 min":1,"5-10 min":2,"> 10 min":3}
    pivot["_ord"] = pivot["Exceed_Bucket"].map(bucket_order).fillna(99)
    pivot = pivot.sort_values("_ord").drop(columns=["_ord"])
    gt_row = {"Exceed_Bucket":"Grand Total"}
    for c in REST_SESSION_COLS: gt_row[c] = int(pivot[c].sum())
    gt_row["Grand Total"] = int(pivot["Grand Total"].sum())
    pivot = pd.concat([pivot, pd.DataFrame([gt_row])], ignore_index=True)
    return pivot

rest_bucket_session = build_bucket_session_pivot(rest_exceed)
rest_sup_count      = build_rest_pivot(rest_exceed, "Supervisor Name")
rest_sup_bucket     = build_bucket_pivot(rest_exceed, "Supervisor Name")
print(f"✓ Rest pivots — Count:{len(rest_sup_count)} Bucket:{len(rest_sup_bucket)}")

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
MET_BG   = "#d4f4e2"; MET_FG  = "#1a5c2a"
WARN_BG  = "#fff3cd"; WARN_FG = "#7a5200"
MISS_BG  = "#fde8ea"; MISS_FG = "#9b1c2a"
HDR_DARK = "#1a3a5c"; HDR_MID = "#1f5c99"
TOT_BG   = "#1a3a5c"
BANNER_C = "#8b0020"
SEC_BADGE_BG = "#e6a817"; SEC_BADGE_FG = "#1a1a1a"
FONT     = "font-family:Arial,sans-serif;font-size:11px;"
TH_S     = (f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;"
            f"white-space:nowrap;text-align:center;"
            f"border:1px solid rgba(255,255,255,0.2);")
TD_S     = f"{FONT}padding:4px 8px;border:1px solid #e8e8e8;white-space:nowrap;"
TD_TOT   = (f"{FONT}padding:4px 8px;border:1px solid rgba(255,255,255,0.15);"
            f"background:{TOT_BG};color:#fff;font-weight:bold;")

CSS = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-size:11px;white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #e8e8e8;
   text-align:left;background:#fff}}
.t tbody tr.tot td{{background:{TOT_BG}!important;color:#fff!important;
   font-weight:bold!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.warn{{background:{WARN_BG}!important;color:{WARN_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.sec-badge{{display:inline-block;font-size:12px;font-weight:bold;
   background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};
   padding:4px 12px;margin:24px 0 4px;border-radius:3px}}
.note{{font-size:10.5px;color:#555;background:#f8f8f8;
   border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;
   margin:0 0 10px;border-radius:0 3px 3px 0}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def fv(v, pct=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "&#8212;"
    if pct: return f"{float(v):.1f}%"
    if isinstance(v, float): return f"{v:,.0f}"
    return str(v)

def adh_color_abs(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("", "")
    fv_ = float(v)
    if fv_ >= ADH_GREEN:    cls, bg, fg = "met",  MET_BG,  MET_FG
    elif fv_ >= ADH_YELLOW: cls, bg, fg = "warn", WARN_BG, WARN_FG
    else:                   cls, bg, fg = "miss", MISS_BG, MISS_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def _th(l, bg=HDR_MID):   return f'<th style="{TH_S}background:{bg};">{l}</th>'
def _thl(l, bg=HDR_DARK): return f'<th style="{TH_S}background:{bg};text-align:left;">{l}</th>'

def _tdl(v, tot=False, em=False):
    val = str(v) if v is not None and not (isinstance(v, float) and np.isnan(v)) else "&#8212;"
    if tot: return f'<td style="{TD_TOT}text-align:left;">{val}</td>'
    return f'<td style="{TD_S}background:#fff;">{val}</td>'

def _tdn(v, tot=False, em=False):
    val = fv(v)
    if tot: return f'<td style="{TD_TOT}text-align:right;">{val}</td>'
    return f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>'

def _sec(num, title, note, em=False):
    bs = (f"display:inline-block;font-size:12px;font-weight:bold;"
          f"background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};"
          f"padding:4px 12px;margin:24px 0 4px;border-radius:3px;")
    ns = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
          f"border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;"
          f"margin:0 0 10px;border-radius:0 3px 3px 0;display:block;")
    if em:
        return (f'<p style="margin:24px 0 4px;">'
                f'<span style="{bs}">{num}. {title}</span></p>'
                f'<p style="{ns}">{note}</p>')
    return (f'<div style="margin:24px 0 4px;">'
            f'<span class="sec-badge">{num}. {title}</span></div>'
            f'<div class="note">{note}</div>')

def spacer(em=False):
    if em:
        return ('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                '<tr><td style="height:24px;font-size:1px;">&nbsp;</td></tr></table>')
    return '<div style="height:24px;"></div>'

# ══════════════════════════════════════════════════════════════════════════════
# TABLE RENDERERS
# ══════════════════════════════════════════════════════════════════════════════
def render_adh_pivot(df, index_col, em=False):
    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    h.append(_thl(index_col))
    for c in ACT_COLS: h.append(_th(c))
    h.append(_th("Grand Total", bg=TOT_BG))
    h.append('</tr></thead><tbody>')
    for i, row in df.iterrows():
        tot = str(row.get(index_col, "")) == "Grand Total"
        h.append('<tr>' if em else f'<tr class="{"tot" if tot else ""}">')
        h.append(_tdl(row.get(index_col), tot, em))
        for c in ACT_COLS + ["Grand Total"]:
            v = row.get(c)
            if isinstance(v, float) and np.isnan(v): v = None
            val = fv(v, pct=True)
            if tot:
                h.append(f'<td style="{TD_TOT}text-align:right;">{val}</td>')
            elif v is None:
                h.append(f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>')
            else:
                cls, inl = adh_color_abs(v, em)
                if em:  h.append(f'<td style="{TD_S}text-align:right;{inl}">{val}</td>')
                else:   h.append(f'<td class="{cls}" style="{TD_S}text-align:right;">{val}</td>')
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

def render_rest_count(df, index_col, em=False):
    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    h.append(_thl(index_col))
    for c in REST_SESSION_COLS: h.append(_th(c))
    h.append(_th("Grand Total", bg=TOT_BG))
    h.append('</tr></thead><tbody>')
    data_rows = df[df[index_col] != "Grand Total"]
    max_vals  = {c: data_rows[c].max() for c in REST_SESSION_COLS}
    for i, row in df.iterrows():
        tot = str(row.get(index_col, "")) == "Grand Total"
        h.append('<tr>' if em else f'<tr class="{"tot" if tot else ""}">')
        h.append(_tdl(row.get(index_col), tot, em))
        for c in REST_SESSION_COLS:
            v   = row.get(c, 0)
            val = fv(v)
            if tot:
                h.append(f'<td style="{TD_TOT}text-align:right;">{val}</td>')
            else:
                mx = max_vals.get(c, 1) or 1
                if v and float(v) > 0:
                    ratio = float(v) / float(mx)
                    if ratio >= 0.7:   bg, fg = MISS_BG, MISS_FG
                    elif ratio >= 0.4: bg, fg = WARN_BG, WARN_FG
                    else:              bg, fg = MET_BG,  MET_FG
                    h.append(f'<td style="{TD_S}text-align:right;background:{bg};'
                             f'color:{fg};font-weight:bold;">{val}</td>')
                else:
                    h.append(f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>')
        h.append(_tdn(row.get("Grand Total"), tot, em))
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

def render_bucket_pivot(df, index_col, em=False):
    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    h.append(_thl(index_col))
    for c in EXCEED_BUCKETS: h.append(_th(c))
    h.append(_th("Grand Total", bg=TOT_BG))
    h.append('</tr></thead><tbody>')
    max_vals = {c: df[df[index_col] != "Grand Total"][c].max() for c in EXCEED_BUCKETS}
    for i, row in df.iterrows():
        tot = str(row.get(index_col, "")) == "Grand Total"
        h.append('<tr>' if em else f'<tr class="{"tot" if tot else ""}">')
        h.append(_tdl(row.get(index_col), tot, em))
        for c in EXCEED_BUCKETS:
            v   = row.get(c, 0)
            val = fv(v)
            if tot:
                h.append(f'<td style="{TD_TOT}text-align:right;">{val}</td>')
            else:
                mx = max_vals.get(c, 1) or 1
                if not (isinstance(v, float) and np.isnan(v)) and v > 0:
                    ratio = float(v) / float(mx)
                    if ratio >= 0.7:   bg, fg = MISS_BG, MISS_FG
                    elif ratio >= 0.4: bg, fg = WARN_BG, WARN_FG
                    else:              bg, fg = MET_BG,  MET_FG
                    if em:
                        h.append(f'<td style="{TD_S}text-align:right;background:{bg};color:{fg};font-weight:bold;">{val}</td>')
                    else:
                        h.append(f'<td style="{TD_S}text-align:right;">'
                                 f'<span style="background:{bg};color:{fg};font-weight:bold;padding:2px 6px;border-radius:3px;">{val}</span></td>')
                else:
                    h.append(f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>')
        h.append(_tdn(row.get("Grand Total"), tot, em))
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

def render_bucket_session(df, em=False):
    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    h.append(_thl("Exceed Bucket"))
    for c in REST_SESSION_COLS: h.append(_th(c))
    h.append(_th("Grand Total", bg=TOT_BG))
    h.append('</tr></thead><tbody>')
    data_rows = df[df["Exceed_Bucket"] != "Grand Total"]
    col_stats  = {}
    for c in REST_SESSION_COLS + ["Grand Total"]:
        vals = pd.to_numeric(data_rows[c], errors="coerce").dropna()
        col_stats[c] = (vals.min(), vals.max()) if len(vals) > 0 else (0, 1)
    for i, row in df.iterrows():
        tot = str(row.get("Exceed_Bucket", "")) == "Grand Total"
        h.append('<tr>' if em else f'<tr class="{"tot" if tot else ""}">')
        h.append(_tdl(row.get("Exceed_Bucket", ""), tot, em))
        for c in REST_SESSION_COLS + ["Grand Total"]:
            v   = row.get(c, 0)
            val = fv(v)
            if tot:
                h.append(f'<td style="{TD_TOT}text-align:right;">{val}</td>')
            elif v and float(v) > 0:
                mn, mx = col_stats[c]
                ratio = 1.0 - ((float(v) - mn) / (mx - mn)) if mx > mn else 0.5
                if ratio >= 0.7:    bg, fg = MET_BG,  MET_FG
                elif ratio >= 0.35: bg, fg = WARN_BG, WARN_FG
                else:               bg, fg = MISS_BG, MISS_FG
                h.append(f'<td style="{TD_S}text-align:right;background:{bg};'
                         f'color:{fg};font-weight:bold;">{val}</td>')
            else:
                h.append(f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>')
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_all(em=False):
    parts = []
    bi  = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    bs  = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    inn = (f'<p style="{bi}">&#128202; Expedia VN — Agent Compliance Report</p>'
           f'<p style="{bs}">As of: <strong>{report_date_s}</strong> &nbsp;|&nbsp; '
           f'Period: {month_start} → {report_date_d} &nbsp;|&nbsp; '
           f'Generated: {report_now.strftime("%Y-%m-%d %H:%M")}</p>')
    if em:
        parts.append(
            f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 14px;">'
            f'<tr><td style="background:{BANNER_C};padding:10px 14px;border-radius:4px;">'
            f'{inn}</td></tr></table>')
    else:
        parts.append(f'<div style="background:{BANNER_C};padding:10px 14px;'
                     f'border-radius:4px;margin:0 0 14px;">{inn}</div>')

    parts.append(_sec("1", "Schedule Adherence — Supervisor Wise",
        f"Adherence Match Rate % by Supervisor × Activity Type. "
        f"&#9632; Green &ge;{ADH_GREEN:.0f}% &nbsp; "
        f"&#9632; Yellow {ADH_YELLOW:.0f}–{ADH_GREEN-0.1:.0f}% &nbsp; "
        f"&#9632; Red &lt;{ADH_YELLOW:.0f}%", em))
    parts.append(render_adh_pivot(adh_sup, "Supervisor Name", em))

    parts.append(spacer(em))
    parts.append(_sec("2", "Schedule Adherence — LOB Wise",
        f"Adherence Match Rate % by LOB × Activity Type. "
        f"&#9632; Green &ge;{ADH_GREEN:.0f}% &nbsp; "
        f"&#9632; Yellow {ADH_YELLOW:.0f}–{ADH_GREEN-0.1:.0f}% &nbsp; "
        f"&#9632; Red &lt;{ADH_YELLOW:.0f}%", em))
    parts.append(render_adh_pivot(adh_lob, "LOB", em))

    parts.append(spacer(em))
    parts.append(_sec("3", "Schedule Adherence — Day Wise (MTD)",
        f"Daily Adherence Match Rate % from {month_start} to {report_date_d}. "
        f"&#9632; Green &ge;{ADH_GREEN:.0f}% &nbsp; "
        f"&#9632; Yellow {ADH_YELLOW:.0f}–{ADH_GREEN-0.1:.0f}% &nbsp; "
        f"&#9632; Red &lt;{ADH_YELLOW:.0f}%", em))
    parts.append(render_adh_pivot(adh_day_pvt, "Date", em))

    parts.append(spacer(em))
    parts.append(_sec("4", "Rest Compliance — Overbreak Count by Supervisor",
        f"Number of sessions exceeding allowed rest time. "
        f"Break limit: 15 min &nbsp; Lunch limit: 60 min.", em))
    parts.append(render_rest_count(rest_sup_count, "Supervisor Name", em))

    parts.append(spacer(em))
    parts.append(_sec("5", "Rest Compliance — Exceed Bucket Distribution",
        f"Breakdown of overbreak instances by duration bucket. "
        f"&#9632; Color intensity = relative frequency.", em))
    parts.append(render_bucket_pivot(rest_sup_bucket, "Supervisor Name", em))

    parts.append(spacer(em))
    parts.append(_sec("6", "Rest Compliance — Bucket × Session Type",
        f"Count of overbreak instances by duration bucket × session type "
        f"(Break_1, Break_2, Lunch). Higher = worse (more red).", em))
    parts.append(render_bucket_session(rest_bucket_session, em))

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    Please find the Expedia VN Agent Compliance Report as of the
    <strong>{day_ord} of {month_yr}</strong>,
    covering Schedule Adherence and Rest Compliance.
</p>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 12px;">
"""

def signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;font-weight:bold;margin:0 0 2px;">BI Associate</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:8px;">
    Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: Schedule_Adherence_Gap.parquet + Agent_Rest_Compliance_Log.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# BUILD RAWDATA ATTACHMENT
# ══════════════════════════════════════════════════════════════════════════════
def build_rawdata_attachment(fmt: str = "xlsx") -> str:
    fname    = f"AgentCompliance_Rawdata_{report_date.strftime('%Y%m%d')}.{fmt}"
    tmp_path = os.path.join(tempfile.gettempdir(), fname)

    def _autofit(ws):
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    if fmt == "csv":
        adh_raw.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    elif fmt == "xlsx":
        with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
            adh.to_excel(writer, index=False, sheet_name="ADH Raw");                  _autofit(writer.sheets["ADH Raw"])
            adh_sup.to_excel(writer, index=False, sheet_name="ADH Sup Pivot");        _autofit(writer.sheets["ADH Sup Pivot"])
            adh_lob.to_excel(writer, index=False, sheet_name="ADH LOB Pivot");        _autofit(writer.sheets["ADH LOB Pivot"])
            adh_day_pvt.to_excel(writer, index=False, sheet_name="ADH Day Pivot");    _autofit(writer.sheets["ADH Day Pivot"])
            rest_exceed.to_excel(writer, index=False, sheet_name="Rest Raw (Exceed)");_autofit(writer.sheets["Rest Raw (Exceed)"])
            rest_sup_count.to_excel(writer, index=False, sheet_name="Rest Count");    _autofit(writer.sheets["Rest Count"])
            rest_sup_bucket.to_excel(writer, index=False, sheet_name="Rest Bucket");  _autofit(writer.sheets["Rest Bucket"])
            rest_bucket_session.to_excel(writer, index=False, sheet_name="Bucket x Session"); _autofit(writer.sheets["Bucket x Session"])
    else:
        raise ValueError(f"Unsupported format: {fmt}")

    size_kb = os.path.getsize(tmp_path) / 1024
    print(f"✓ Attachment ready: {fname} ({size_kb:.1f} KB)")
    return tmp_path

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb  = ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
           f"<style>{CSS}</style></head><body>"
           + build_all(em=False) + "</body></html>")
    esc = nb.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{esc}" style="width:100%;border:none;min-height:900px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'))
    print("✓ Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    html_body = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + greeting() + build_all(em=True) + signature() + "</div>"
    )

    def send_auto(to, cc, subject, html_body, attachments=None, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe"
                     for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol   = win32com.client.Dispatch("Outlook.Application")
            ns   = ol.GetNamespace("MAPI"); ns.Logon()
            mail = ol.CreateItem(0)
            mail.To       = to
            mail.CC       = cc
            mail.Subject  = subject
            mail.HTMLBody = html_body
            if attachments:
                for fpath in attachments:
                    abs_path = os.path.abspath(fpath)
                    if os.path.exists(abs_path):
                        mail.Attachments.Add(abs_path)
                        print(f"✓ Attached: {os.path.basename(abs_path)}")
                    else:
                        print(f"⚠️  Not found: {abs_path}")
            mail.Send()
            print(f"✓ Sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    attachments = []
    if ATTACH_RAWDATA:
        attach_path = build_rawdata_attachment(fmt=ATTACH_FORMAT)
        attachments.append(attach_path)

    send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, html_body,
              attachments=attachments, quit_after=True)

    for f in attachments:
        try: os.remove(f); print(f"✓ Cleaned: {f}")
        except: pass

✓ Report date : 2026-08-19
✓ Month       : 26_08
✓ Subject     : Expedia VN - Agent Compliance Report - as of the 19th of Aug'26
📂 Loading Schedule_Adherence_Gap...
✓ Adherence rows MTD to 2026-08-19: 8615
✓ Adherence pivots — Sup:8 LOB:3 Day:20
📂 Loading Agent_Rest_Compliance_Log...
✓ Rest exceeding rows MTD to 2026-08-19: 1451
✓ Rest pivots — Count:8 Bucket:8


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Supervisor Name,Break,Lunch,Training | Coaching,Grand Total
Ann,49.9%,85.4%,73.6%,73.5%
Chau Thien Kim,46.1%,64.4%,48.3%,52.0%
Mia Minh Le,54.1%,77.2%,16.7%,68.0%
Nguyen Thi Anh Thu,36.2%,69.8%,43.7%,52.2%
Tran Hoang My Anh,38.4%,71.3%,56.5%,59.6%
Tran Thao Uyen,37.9%,70.2%,53.1%,59.0%
Truong Thien Thanh Toan,38.0%,73.6%,46.7%,60.7%
Grand Total,41.1%,72.0%,48.6%,57.1%
LOB,Break,Lunch,Training | Coaching,Grand Total
Lodging,41.2%,71.0%,48.1%,56.1%


✓ Display done
✓ Attachment ready: AgentCompliance_Rawdata_20260819.xlsx (1041.1 KB)
✓ Attached: AgentCompliance_Rawdata_20260819.xlsx
✓ Sent → puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com
✓ Cleaned: C:\Users\HUUCHI~1.NGU\AppData\Local\Temp\AgentCompliance_Rawdata_20260819.xlsx


In [2]:
# import io, time, tempfile, os, win32clipboard
# import numpy as np
# from PIL import Image
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import StaleElementReferenceException
# from webdriver_manager.chrome import ChromeDriverManager

# # ══════════════════════════════════════════════════════════════════════════════
# # CONFIG — TEAMS
# # ══════════════════════════════════════════════════════════════════════════════
# SEND_TEAMS       = True
# TEAMS_GROUP_NAME = "Work together"
# RENDER_WIDTH     = 1000

# # ══════════════════════════════════════════════════════════════════════════════
# # STEP 1 — Render HTML → PNG dùng Chrome headless (không cần Playwright)
# # ══════════════════════════════════════════════════════════════════════════════
# def html_to_pil_image(html_content: str, width: int = 1000) -> Image.Image:
#     """
#     Ghi HTML ra file tạm → mở bằng Chrome headless → chụp full-page screenshot.
#     Không dùng Playwright → tránh hoàn toàn asyncio conflict trong Jupyter.
#     """
#     # Ghi HTML ra file tạm
#     full_html = (
#         "<!DOCTYPE html><html><head><meta charset='utf-8'>"
#         f"<style>{CSS}"
#         "body{margin:0;padding:20px;background:#fff;}"
#         "</style></head><body>"
#         + html_content
#         + "</body></html>"
#     )
#     tmp_html = os.path.join(tempfile.gettempdir(), "report_preview.html")
#     with open(tmp_html, "w", encoding="utf-8") as f:
#         f.write(full_html)

#     # Khởi động Chrome headless
#     options = Options()
#     options.add_argument("--headless=new")
#     options.add_argument(f"--window-size={width},900")
#     options.add_argument("--hide-scrollbars")
#     options.add_argument("--disable-gpu")
#     options.add_argument("--no-sandbox")
#     options.add_argument("--disable-dev-shm-usage")

#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()),
#         options=options
#     )

#     try:
#         driver.get(f"file:///{tmp_html.replace(os.sep, '/')}")
#         time.sleep(2)

#         # Lấy chiều cao thực tế của content
#         height = driver.execute_script("return document.body.scrollHeight")
#         driver.set_window_size(width, height + 40)
#         time.sleep(0.5)

#         # Chụp screenshot dạng PNG bytes
#         png_bytes = driver.get_screenshot_as_png()
#     finally:
#         driver.quit()

#     img = Image.open(io.BytesIO(png_bytes))
#     print(f"✓ Rendered: {img.size[0]}×{img.size[1]}px")
#     return img

# # ══════════════════════════════════════════════════════════════════════════════
# # STEP 2 — Copy PIL Image vào Windows Clipboard
# # ══════════════════════════════════════════════════════════════════════════════
# def copy_image_to_clipboard(image: Image.Image):
#     output = io.BytesIO()
#     image.convert("RGB").save(output, "BMP")
#     data = output.getvalue()[14:]
#     output.close()
#     win32clipboard.OpenClipboard()
#     win32clipboard.EmptyClipboard()
#     win32clipboard.SetClipboardData(win32clipboard.CF_DIB, data)
#     win32clipboard.CloseClipboard()
#     print("✓ Image copied to clipboard")

# # ══════════════════════════════════════════════════════════════════════════════
# # STEP 3 — Selenium paste ảnh vào Teams group chat
# # ══════════════════════════════════════════════════════════════════════════════
# def paste_image_to_teams(group_name: str, image: Image.Image, caption: str):
#     copy_image_to_clipboard(image)

#     # Chrome với profile đã login Teams
#     chrome_options = Options()
#     chrome_options.add_argument(r"--user-data-dir=C:/temp/new_chrome_profile")
#     chrome_options.add_argument(r"--profile-directory=Default")
#     chrome_options.add_argument("--start-maximized")

#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()),
#         options=chrome_options
#     )
#     wait = WebDriverWait(driver, 25)

#     try:
#         driver.get("https://teams.microsoft.com/")
#         time.sleep(8)

#         # Tìm và click group chat
#         clicked = False
#         for attempt in range(3):
#             try:
#                 chat = wait.until(EC.element_to_be_clickable(
#                     (By.XPATH, f"//span[contains(text(),'{group_name}')]")))
#                 chat.click()
#                 clicked = True
#                 print(f"✓ Entered group: {group_name}")
#                 break
#             except StaleElementReferenceException:
#                 time.sleep(2)
#             except Exception as e:
#                 print(f"❌ Cannot find group '{group_name}': {e}")
#                 return

#         if not clicked:
#             print(f"❌ Failed after 3 attempts")
#             return

#         time.sleep(3)

#         # Click chat box
#         chat_box = wait.until(EC.element_to_be_clickable(
#             (By.CSS_SELECTOR, "div[role='textbox']")))
#         chat_box.click()

#         # Gõ caption
#         ActionChains(driver).send_keys(caption).perform()
#         time.sleep(1)

#         # Paste ảnh từ clipboard
#         ActionChains(driver).key_down(Keys.CONTROL).send_keys("v").key_up(Keys.CONTROL).perform()
#         print("⏳ Uploading image...")
#         time.sleep(6)

#         # Gửi
#         ActionChains(driver).send_keys(Keys.ENTER).perform()
#         print("✓ Message sent to Teams!")
#         time.sleep(3)

#     except Exception as e:
#         print(f"❌ Error: {e}")
#     finally:
#         driver.quit()
#         print("✓ Chrome closed")

# # ══════════════════════════════════════════════════════════════════════════════
# # MAIN
# # ══════════════════════════════════════════════════════════════════════════════
# if SEND_TEAMS:
#     print("🎨 Rendering HTML report via Chrome headless...")
#     img = html_to_pil_image(build_all(em=False), width=RENDER_WIDTH)

#     caption_lines = [
#         f"📊 Agent Compliance Report — updated to {report_date_s}",
#     ]

#     copy_image_to_clipboard(img)

#     chrome_options = Options()
#     chrome_options.add_argument(r"--user-data-dir=C:/temp/new_chrome_profile")
#     chrome_options.add_argument(r"--profile-directory=Default")
#     chrome_options.add_argument("--start-maximized")

#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()),
#         options=chrome_options
#     )
#     wait = WebDriverWait(driver, 25)

#     try:
#         driver.get("https://teams.microsoft.com/")
#         time.sleep(8)

#         clicked = False
#         for attempt in range(3):
#             try:
#                 chat = wait.until(EC.element_to_be_clickable(
#                     (By.XPATH, f"//span[contains(text(),'{TEAMS_GROUP_NAME}')]")))
#                 chat.click()
#                 clicked = True
#                 print(f"✓ Entered group: {TEAMS_GROUP_NAME}")
#                 break
#             except StaleElementReferenceException:
#                 time.sleep(2)
#             except Exception as e:
#                 print(f"❌ Cannot find group: {e}")
#                 break

#         if not clicked:
#             print("❌ Failed"); driver.quit()
#         else:
#             time.sleep(3)
#             chat_box = wait.until(EC.element_to_be_clickable(
#                 (By.CSS_SELECTOR, "div[role='textbox']")))
#             chat_box.click()

#             actions = ActionChains(driver)
#             for i, line in enumerate(caption_lines):
#                 actions.send_keys(line)
#                 if i < len(caption_lines) - 1:
#                     actions.key_down(Keys.SHIFT).send_keys(Keys.ENTER).key_up(Keys.SHIFT)
#             actions.perform()
#             time.sleep(1)

#             ActionChains(driver).key_down(Keys.CONTROL).send_keys("v").key_up(Keys.CONTROL).perform()
#             print("⏳ Uploading image...")
#             time.sleep(6)

#             ActionChains(driver).send_keys(Keys.ENTER).perform()
#             print("✓ Sent!")
#             time.sleep(3)

#     except Exception as e:
#         print(f"❌ Error: {e}")
#     finally:
#         driver.quit()
#         print("✓ Chrome closed")